In [1]:
# !pip install resemblyzer

In [553]:
import numpy as np
import pandas as pd
import librosa
import joblib

#For transcription
from faster_whisper import WhisperModel
from keras.models import load_model
from pydub import AudioSegment
import re
from statistics import mode

#For Speaker Diarization
from resemblyzer import VoiceEncoder, preprocess_wav
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import AgglomerativeClustering

#For Spekear Identification (Classification)
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

import warnings
warnings.filterwarnings('ignore')

# Input Audio File + Transcription

Normalization and Audio boosting will be done to help improve the embedding results of resemblyzer

The waveform of the audio file will be obtained using the Librosa library. Then the faster whisper model will be used to provide the initial transcript with some segmentation

In [554]:
#Convert mp files to wav since resemblyzer embedding works better with wav
def convert_to_wav(input_file, output_file):
    sound = AudioSegment.from_file(input_file)
    sound = sound.set_frame_rate(16000).set_channels(1)
    sound.export(output_file, format = "wav")

In [555]:
def normalize_audio(waveform):
    peak = np.max(np.abs(waveform))
    if peak > 0:
        waveform = waveform / peak
    return waveform

In [556]:
def boost_audio(waveform, db):
    factor = 10 ** (db / 20) 
    boosted = waveform * factor
    boosted = np.clip(boosted, -1.0, 1.0) 
    return boosted

In [557]:
def import_audio_file(file):
    #Extracting audio file's waveform with librosa
    file_path = file
    waveform, sample_rate = librosa.load(file_path, res_type = "kaiser_fast", sr = None)
    return (waveform, sample_rate)

file = "./SampleAudio/MSK0046.mp3"
waveform, sample_rate = import_audio_file(file)
sample_rate

16000

In [558]:
#Transcribe text using faster whisper
def transcribe_text(file, model_size = "medium.en", device = "cuda", compute_type = "float16"):
    #Initiate whisper model
    whisper_model = WhisperModel(model_size, device = device, compute_type = compute_type)
    
    segments, _ = whisper_model.transcribe(file,
                                           vad_filter = True,
                                           word_timestamps = True, 
                                           beam_size = 8)
    return segments
    
segments = transcribe_text(file)

In [559]:
# for segment in segments:
#     for word in segment.words:
#         print(f"{word.start}: {word.word} \n")

# Speaker Diarization

Speaker Diarization/Differentiation will be done by doing the following steps

1.) Using resemblyzer to embedd each spoken word from the audio. This embedding is the extracted features, represetning the voice characteristics of each spoken word and will be used to differentiate between speakers

2.) Performing Clustering on the extracted embedded features

## Extracting embedding features

In [560]:
def extract_embeddings(segments, waveform, sample_rate):
    #Generate embeddings for each audio word usig resemblyzer
    print("Extracting embeddings, Please Wait...")
    encoder = VoiceEncoder()
    embeddings = []
    complete_transcript = []
    start_times = []
    end_times = []
    
    for segment in segments:
        #Extracting embeddings for each individual words
        for word in segment.words:   
            #Get the start and end index of the segment
            start_index = int(word.start * sample_rate)
            end_index = int(word.end * sample_rate)

            start_times.append(word.start)
            end_times.append(word.end)
            
            #Get the complete waveform of the current segment
            segment_waveform = waveform[start_index:end_index]
        
            #Extract embeddings
            segment_embeddings = encoder.embed_utterance(segment_waveform)
            embeddings.append(segment_embeddings)
    
            complete_transcript.append(word.word)

    #Fail check
    if len(embeddings) == 0:
        raise ValueError("Failed to detect any voices. Please upload a new audio file")

    return embeddings, complete_transcript, start_times, end_times

embeddings, complete_transcript, start_times, end_times = extract_embeddings(segments, waveform, sample_rate)
print("Done Extracting Embeddings")

Extracting embeddings, Please Wait...
Loaded the voice encoder model on cpu in 0.01 seconds.
Done Extracting Embeddings


## Clustering using Agglomerative Clustering

Agglomerative clustering was the chosen clustering technique because of its advantage of being able to generate clusters without knowing the K value. This is important as an audio file may contain an unknown number of speakers.

In [561]:
#Setting the distance threshold automatically 
def calculate_threshold(embeddings):  
    Z = linkage(embeddings, method = 'ward', metric = 'euclidean')
    distances = Z[:, 2]
    differences = np.diff(distances)
    
    #getting the index with the largest difference
    optimal_index = np.argmax(differences)
    
    #Selecting the threshold based on the index with the largest difference
    buffer = 0.25 #Add some buffer to help avoid over-splitting (having more clusters)
    threshold = distances[optimal_index]
    threshold = threshold + buffer

    return threshold

threshold = calculate_threshold(embeddings)
print("Auto-selected threshold:", threshold)

Auto-selected threshold: 4.82177823409922


In [562]:
def apply_clustering(embeddings, threshold):
    agglo_cluster = AgglomerativeClustering(n_clusters = None, #Setting to None because using the calculated distance threhsold instead
                                            metric = "euclidean",
                                            linkage = "ward",
                                            distance_threshold = threshold)
    
    cluster_pred = agglo_cluster.fit_predict(embeddings)
    return cluster_pred

cluster_pred = apply_clustering(embeddings, threshold)
cluster_pred

array([0, 0, 0, ..., 0, 1, 1], dtype=int64)

In [563]:
#Store the transcript and cluster prediction to a df
def create_cluster_df(cluster_pred, complete_transcript, start_times, end_times):
    df_word_transcript = pd.DataFrame(columns = ["Transcripts", "Prediction", "StartTime", "EndTime"])
    df_word_transcript["StartTime"] = start_times
    df_word_transcript["EndTime"] = end_times
    df_word_transcript["Prediction"] = cluster_pred
    df_word_transcript["Transcripts"] = complete_transcript

    return df_word_transcript

df_word_transcript = create_cluster_df(cluster_pred, complete_transcript, start_times, end_times)
df_word_transcript.head(10)

,Transcripts,Prediction,StartTime,EndTime
0,what,0,0.00,0.16
1,brings,0,0.16,0.44
2,in,0,0.44,0.62
3,here,0,0.62,0.78
4,today.,0,0.78,1.54
5,I'm,1,2.58,2.90
6,here,1,2.90,3.10
7,because,1,3.10,3.48
8,my,1,3.48,3.78
9,left,1,3.78,4.12


## Converting the transcript from word per word back to sentence form

In [564]:
def recreate_sentences(df_word_transcript):
    sentences = []
    current_words = []
    current_predictions = []
    sentence_start_time = None
    sentence_end_time = None
    
    #Iterate through each row of the original dataframe
    for indx, row in df_word_transcript.iterrows():
        word = row["Transcripts"]
        pred = row["Prediction"]
        start_time = row["StartTime"]
        end_time = row["EndTime"]
    
        #Append the current word 
        current_words.append(word)
        current_predictions.append(pred)

        #Record start time if the word is the first word of a sentence
        if sentence_start_time is None:
            sentence_start_time = start_time
            
        #Start new sentence if the current word contains a punctuation
        if re.search(r"[.!?]", word):
        
            #Exclude title words such as Mr. Ms. Mrs. Dr.
            if word in [" Mr.", " Ms.", " Mrs.", " Dr."]:
                continue
    
            #Form the current sentence
            sentence_final = " ".join(current_words)
            #Using mode to obtain the final prediction of which cluster spoke the sentence
            prediction_final = mode(current_predictions)

            #Record the end time of the last word of the sentence
            sentence_end_time = end_time
    
            sentences.append({
                "Speaker": prediction_final,
                "Transcripts": sentence_final,
                "StartTime": sentence_start_time,
                "EndTime": sentence_end_time
            })
    
            #Reseting for the next sentence
            current_words = []
            current_predictions = []
            sentence_start_time = None
            sentence_end_time = None
    
    df_sentence_transcript = pd.DataFrame(sentences)
    return(df_sentence_transcript)

df_sentence_transcript = recreate_sentences(df_word_transcript)
df_sentence_transcript.head(20)

,Speaker,Transcripts,StartTime,EndTime
0,0,what brings in here today.,0.00,1.54
1,1,I'm here because my left hand kind of ...,2.58,23.76
2,0,Okay and how long has this been going ...,25.02,26.60
3,1,So for the past two days but it got ...,29.02,34.92
4,0,"Okay and have you had any, if you wer...",37.10,42.96
5,1,"It's just at the base of my thumb, yo...",44.82,51.98
6,0,Okay yeah so just over there.,53.24,56.82
7,0,Okay and what kind of pain is it?,57.82,59.90
8,0,Is it sharp or is it aching?,59.96,61.50
9,1,At baseline it's an achy pain but if ...,66.74,81.34


# Speaker Identification

Labelling which cluster is the doctor and patient using the trained Bidirectional-LSTM model from the Classification notebook

In [545]:
#Combine all transcripts within each clusters into a single string
def string_cluster(df_sentence_transcript):
    cluster_transcripts = {}
    for cluster in df_sentence_transcript["Speaker"].unique():
        cluster_transcripts[cluster] = ""
    
    for idx, row in df_sentence_transcript.iterrows():
        current_label = row["Speaker"]
        cluster_transcripts[current_label] += row["Transcripts"] + ""
    
    #Put the combined trnascripts into a df
    cluster_transcripts = list(cluster_transcripts.items())
    df_prediction = pd.DataFrame(cluster_transcripts, columns = ["cluster", "transcript"])
    return(df_prediction)

df_prediction = string_cluster(df_sentence_transcript)
df_prediction

,cluster,transcript
0,0,what brings in here today. Okay and hav...
1,1,I'm here because my left hand kind of ...
2,2,Okay and how long has this been going ...


In [546]:
#Perform classification using the trained LSTM model on each string cluster
def label_speaker(row):
    #Create a new dataframe
    le = joblib.load("./SavedModels/LabelEncoder.bin")
    lstm_model = load_model("./SavedModels/lstm_model.keras")
    tokenizer = joblib.load("./SavedModels/Tokenizer.bin")
    
    #Perform the text processing
    pred_text = text_clean(row["transcript"])
    
    #Preprocess the text
    pred_seq = tokenizer.texts_to_sequences([pred_text])
    max_length = 500
    pred_padded = pad_sequences(pred_seq, maxlen = max_length, padding ='post')
    
    #Predict with the lstm model
    final_prediction = lstm_model.predict(pred_padded)
    #Also get the prediction probability incase that no doctor was predicted
    prediction_prob = float(final_prediction[0][0])
    final_prediction = (final_prediction > 0.5).astype("int")[0]
    
    row["SpeakerLabel"] = le.inverse_transform(final_prediction)[0]
    row["PatientPredictProbability"] = prediction_prob
    
    return row

df_prediction = df_prediction.apply(label_speaker, axis = 1)
df_prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step


,cluster,transcript,SpeakerLabel,PatientPredictProbability
0,0,what brings in here today. Okay and hav...,Doctor,0.051426
1,1,I'm here because my left hand kind of ...,Patient,0.881347
2,2,Okay and how long has this been going ...,Doctor,0.098004


In [547]:
#At least one doctor and one patient must be identified.
def detect_missing_identities(df):
    #In case no doctor was idenfied, the cluster with lowest patient probability will be labelled as doctor
    if "Doctor" not in df["SpeakerLabel"].values:
        #Pick the row with the lowest patient probability
        chosen_row = 0
        lowest_probability = 1
        for idx, row in df.iterrows():
            if row["PatientPredictProbability"] < lowest_probability:
                lowest_probability = row["PatientPredictProbability"]
                chosen_row = idx
        #Change the chosen row speaker into doctor
        df.loc[chosen_row, "SpeakerLabel"] = "Doctor"
    
    #In case of no patient, identify highest patient probability
    if "Patient" not in df["SpeakerLabel"].values:
        #Pick the row with the lowest patient probability
        chosen_row = 0
        highest_probability = 0
        for idx, row in df.iterrows():
            if row["PatientPredictProbability"] > highest_probability:
                highest_probability = row["PatientPredictProbability"]
                chosen_row = idx
        #Change the chosen row speaker into Patient
        df.loc[chosen_row, "SpeakerLabel"] = "Patient"

    return df
    
df_prediction = detect_missing_identities(df_prediction)

In [548]:
#Multiple Patients/Doctors will be numbered e.g. Patient 1, Patient 2
def detect_multiple_identities(df):
    freq = df["SpeakerLabel"].value_counts()
    counts = {}
    counted_labels = []
    
    for label in df["SpeakerLabel"]:
        #Only count multiple identities
        if freq[label] > 1:
            counts[label] = counts.get(label, 0) + 1
            counted_labels.append(f"{label} {counts[label]}")
        else:
            counted_labels.append(label)
    
    df["SpeakerLabel"] = counted_labels
    return df

df_prediction = detect_multiple_identities(df_prediction)

In [549]:
#Replace the unlabelled clusters in the df_sentence_transcript with the predicted labels
def replace_labels(df_sentence_transcript, df_prediction):
    labelled_map = dict(zip(df_prediction["cluster"], df_prediction["SpeakerLabel"]))
    df_sentence_transcript["Speaker"] = df_sentence_transcript["Speaker"].map(labelled_map)
    return df_sentence_transcript

df_sentence_transcript = replace_labels(df_sentence_transcript, df_prediction)
df_sentence_transcript

,Speaker,Transcripts,StartTime,EndTime
0,Doctor 1,what brings in here today.,0.00,1.54
1,Patient,I'm here because my left hand kind of ...,2.58,23.76
2,Doctor 2,Okay and how long has this been going ...,25.02,26.60
3,Patient,So for the past two days but it got ...,29.02,34.92
4,Doctor 1,"Okay and have you had any, if you wer...",37.10,42.96
...,...,...,...,...
86,Doctor 1,You are fairly young but if there was ...,542.80,601.40
87,Doctor 1,Um but uh first yeah let's just do so...,601.40,614.86
88,Patient,Okay yeah that sounds great thank you.,615.22,617.12
89,Doctor 1,You're welcome take care.,617.78,618.82


In [469]:
df_sentence_transcript.to_csv("transcript.csv", index = False)

# Text Cleaning Function

In [ ]:
def text_clean(text):
    #Lower Casing
    text_clean = text.lower()
    
    #Remove punctuations
    punctuations = r"[^\w\s]"
    text_clean = re.sub(punctuations, " ", text_clean)

    #Tokenization
    text_clean = word_tokenize(text_clean)

    #Remove stop words
    stop_words = set(stopwords.words("english"))
    text_clean = [word for word in text_clean if word not in stop_words]

    #Lemmatization
    wn = nltk.WordNetLemmatizer()
    text_lemmatized = []
    for word in text_clean:
        text_lemmatized.append(wn.lemmatize(word))

    #Remove filler words like "um" "like" and basic greetings like "hi", "hey", "hello"
    filtered_list = []
    for token in text_lemmatized:
        if token in ["um", "like", "uhm", "uh", "hi", "hey", "hello"]:
            continue
        #also remove digits
        if token.isdigit():
            continue
            
        filtered_list.append(token)

    text_clean = filtered_list

    #Concat all tokens into a single string
    text_clean = " ".join(text_clean)

    return text_clean

# Wrapping to one function

In [506]:
def med_scribe(file_path, num_speakers = None): #num_speakers = None if unknown
    #Convert to wav
    convert_to_wav(file_path, "output.wav")

    #Extract waveform and sample rate
    waveform, sample_rate = import_audio_file("output.wav")
    waveform = normalize_audio(waveform)
    waveform = boost_audio(waveform, db = 6)

    #Perform transcription
    print("Performing Transcription...")
    segments = transcribe_text(file_path)

    ###Perform Speaker Diarization
    embeddings, complete_transcript, start_times, end_times = extract_embeddings(segments, waveform, sample_rate)
    print("Done Extracting Embeddings")
    print("Performing Speaker Diarization...")
    threshold = calculate_threshold(embeddings)
    cluster_pred = apply_clustering(embeddings, threshold)
    df_word_transcript = create_cluster_df(cluster_pred, complete_transcript, start_times, end_times)

    ###Perform Speaker Identification
    print("Performing Speaker Identification...")
    df_sentence_transcript = recreate_sentences(df_word_transcript)
    df_prediction = string_cluster(df_sentence_transcript)
    df_prediction = df_prediction.apply(label_speaker, axis = 1)

    ###Check for any inconsistencies such as missing doctor/patient identities and multiple identities
    df_prediction = detect_missing_identities(df_prediction)
    df_prediction = detect_multiple_identities(df_prediction)

    #Wrap to one final df
    df_sentence_transcript = replace_labels(df_sentence_transcript, df_prediction)
    
    return df_sentence_transcript

file = "./SampleAudio/MSK0046.mp3"

df_transcript = med_scribe(file)
df_transcript

Performing Transcription...
Extracting embeddings, Please Wait...
Loaded the voice encoder model on cpu in 0.01 seconds.
Done Extracting Embeddings
Performing Speaker Diarization...
Performing Speaker Identification...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step


,Speaker,Transcripts,StartTime,EndTime
0,Doctor,what brings in here today.,0.00,1.54
1,Patient,I'm here because my left hand kind of ...,2.58,23.76
2,Doctor,Okay and how long has this been going ...,25.02,26.60
3,Patient,So for the past two days but it got ...,29.02,34.92
4,Doctor,"Okay and have you had any, if you wer...",37.10,42.96
...,...,...,...,...
86,Doctor,You are fairly young but if there was ...,542.80,601.40
87,Doctor,Um but uh first yeah let's just do so...,601.40,614.86
88,Patient,Okay yeah that sounds great thank you.,615.22,617.12
89,Doctor,You're welcome take care.,617.78,618.82


# GUI w/ Gradio

In [471]:
import gradio as gr

In [472]:
# def get_transcripts(audio):
#     if audio is None:
#         return "", gr.update(visible = False)
    
#     #Process the audio to get the dataframe transcript
#     df_transcript = med_scribe(audio)

#     #Get each line one by one
#     lines = []
#     for idx, row in df_transcript.iterrows():
#         current_line = f"{row["Speaker"]}: {row["Transcripts"]}"

#         row_line = gr.Markdown(current_line)
#         row_button = gr.Button("Action")

#         row_components = gr.Row([row_line, row_button])
#         lines.append(row_components)
            
#     # lines = "\n\n".join(lines)
#     return lines, gr.update(visible = audio)

# #Make the re-upload button apear when an audio file is currently uploaded
# def print_transcripts(transcript):
#     print("True")
#     return gr.update(visible = audio)

HTML not working

In [473]:

# from pydub import AudioSegment
# import uuid


# with gr.Blocks(
#     title="MedScribe",
#     js="""
# function sendTimestamp(ts) {
#     const textbox = gradioApp().querySelector('textarea[aria-label="selected_ts"]');
#     textbox.value = ts;
#     textbox.dispatchEvent(new Event("input"));
    
#     const btn = gradioApp().querySelector('button[aria-label="trigger"]');
#     btn.click();
# }
# """
# ) as demo:



#     def slice_audio(audio_path, ts):
#         start = float(ts)
#         end = start + 5  # or whatever duration you want
    
#         audio = AudioSegment.from_file(audio_path)
#         segment = audio[start*1000:end*1000]
    
#         out = f"segment_{uuid.uuid4().hex}.wav"
#         segment.export(out, format="wav")
#         return out
    


#     def get_transcripts(audio):
#         if audio is None:
#             return "", gr.update(visible = False)
        
#         #Process the audio to get the dataframe transcript
#         df_transcript = med_scribe(audio)
        
#         #Get each line one by one
#         transcript_lines = []
#         for idx, row in df_transcript.iterrows():
#             current_line = f"{row["Speaker"]}: {row["Transcripts"]}"
#             seconds = 5
#             ts = "00:5"
            
            
#             # row_line = gr.Markdown(current_line)
#             # row_button = gr.Button("Action")
        
#             # row_components = gr.Row([row_line, row_button])
#             transcript_lines.append(f"""
#             <div style='display:flex; align-items:center; margin-bottom:8px;'>
#                 <button onclick="sendTimestamp({seconds})" style='margin-right:10px;'>
#                     {ts}
#                 </button>
#                 <span><b>{row['Speaker']}:</b> {row['Transcripts']}</span>
#             </div>
#             """)

#         transcript_lines = "\n\n".join(transcript_lines)
#         return transcript_lines, gr.update(visible = audio)

    
#     with gr.Row():
#         audio = gr.Audio(type = "filepath",
#                 sources = ["upload"], #Only audio import no mic
#                 interactive = True,
#                 editable = False)

#     with gr.Row():
#         #Button to re-upload
#         reupload_button = gr.ClearButton(components = [audio], 
#                                          value = "Re-upload Audio",
#                                          visible = False)

#     transcript_container = gr.HTML()
    
#     audio.change(get_transcripts, 
#                  inputs = audio, 
#                  outputs = [transcript_container, reupload_button])

# #Launch the GUI
# demo.launch(share = True, debug = True, inbrowser=True)

In [516]:
css = """
/* Gr.Audio component styling */
.hf-audio {
    background: rgba(0, 123, 255, 0.15) !important;
    border-radius: 14px !important;
    padding: 20px !important;
    box-shadow: 0 4px 18px rgba(0,0,0,0.25) !important;
    border: 1px solid #333 !important;
}
.hf-audio label {
    color: rgba(0, 123, 255, 1.0) !important;
    font-size: 16px !important;
    font-weight: 700 !important;
}

/* Increasing the size of the timestamps */
#time.svelte-1ffmt2w,
#duration.svelte-1ffmt2w {
    font-size: 22px !important;
    font-weight: 600 !important;
    padding-top: 12px !important;
    border-radius: 8px !important;
}


.hf-audio button {
    background-color: #1e1e1e !important;   
    border-color: #1e1e1e !important;       
    color: white !important;               
}



/* Transcription box */
.transcript-box * {
    font-size: 22px !important;
    line-height: 1.65 !important;
    padding: 20px !important;
    background: rgba(0, 123, 255, 0.15) !important;
    border-radius: 10px !important;
}



"""


In [565]:
def print_transcripts(audio):
    if audio is None:
        return "", gr.update(visible = False)
    
    #Process the audio to get the dataframe transcript
    df_transcript = med_scribe(audio)

    #Get each line one by one
    lines = []
    for idx, row in df_transcript.iterrows():
        #Get the start time of the sentence
        sentence_start_time = round(row["StartTime"])
        minutes = sentence_start_time // 60
        seconds = sentence_start_time % 60
        
        current_line = f"[{minutes}:{seconds:02d}] {row["Speaker"]}: {row["Transcripts"]}"
        lines.append(current_line)

    lines = "\n\n".join(lines)
    return lines, gr.update(visible = audio)

#Make the re-upload button apear when an audio file is currently uploaded
def show_reupload(audio):
    return gr.update(visible = audio)

In [566]:
with gr.Blocks(title = "MedScribe", css = css) as demo:
    ### A heading for the title of the application
    gr.HTML("""
    <div style = "
        text-align: center;
        font-family: 'Segoe UI', sans-serif;
        padding: 0 4px 20px 4px;
        line-height: 1.45;
    ">
        <h1 style = "
            font-size: 28px;
            font-weight: 700;
            margin: 0 0 6px 0;
            display: inline-flex;
            align-items: center;
            gap: 10px;
        ">
            <span>🏥 Medscribe</span>
        </h1>
        <div style = 
            "font-size: 16px; 
             opacity: 0.7; 
             margin-bottom: 12px;
        ">
            Transcribes doctor-patient audio consultations with speaker diarization and speaker identification
        </div>
    """)

    with gr.Row():
        with gr.Column(scale = 1):
            audio = gr.Audio(type = "filepath",
                    label = "Audio File",
                    sources = ["upload"], #Only audio import no mic
                    interactive = True,
                    editable = False,
                    scale = 1,
                    elem_classes = ["hf-audio "]
                    )

            #Button to re-upload
            reupload_button = gr.ClearButton(components = [audio], 
                                             value = "Re-upload Audio",
                                             visible = False,
                                             elem_classes = ["hf-audio "])

        with gr.Column(scale = 2):
            transcript_markdown = gr.Markdown(label = "Transcript", 
                                              height = 1000,
                                              elem_classes = ["transcript-box"])
            
        audio.change(print_transcripts, 
                     inputs = audio, 
                     outputs = [transcript_markdown, reupload_button])

#Launch the GUI
demo.launch(share = True, debug = True, inbrowser = True)

* Running on local URL:  http://127.0.0.1:7885

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\krist\anaconda3\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\krist\anaconda3\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


Performing Transcription...
Extracting embeddings, Please Wait...
Loaded the voice encoder model on cpu in 0.01 seconds.
Done Extracting Embeddings
Performing Speaker Diarization...
Performing Speaker Identification...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 